<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 4.3: Common Pass Idioms
**Prev: [FIRRTL AST Traversal](4.2_firrtl_ast_traversal.ipynb)**<br>
**Next: [A FIRRTL Transform Example](4.4_firrtl_add_ops_per_module.ipynb)**

In this section we look at the typical tasks that programs working on a FIRRTL AST perform: *collecting* information, *adding* statements and *deleting* statements. We will use the AST from the previous section.

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._
import chisel3.reflect.DataMirror

The IR, the parser (`parseCircuit`), `serialize` and the traversal helpers (`allStatements`, `directExprs`, `allExprs`) that we wrote in Module 4.2 are needed again. Notebooks do not share definitions, so the next cell simply contains all of that code once more. Run it, and skip over it:

In [ ]:
// The FIRRTL library of Modules 4.2: IR, parser, serializer and traversal helpers.
// Nothing new here: just run this cell.
// ---- Expressions ----
sealed trait Expression {
  // Apply f to the child Expressions of this node (like firrtl's `map`).
  def mapExpr(f: Expression => Expression): Expression = this match {
    case DoPrim(op, args, consts) => DoPrim(op, args.map(f), consts)
    case leaf                     => leaf
  }
}
case class Reference(name: String) extends Expression                      // x, io.in[0], m0.io.out
case class Literal(tpe: String, value: String) extends Expression          // UInt<3>(0h5)
case class DoPrim(op: String, args: Seq[Expression], consts: Seq[Int]) extends Expression // add(a, b), bits(a, 3, 0), mux(c, a, b)

// ---- Statements ----
sealed trait Statement {
  def info: String // source locator, e.g. "cmd3.sc 6:19" ("" if there is none)

  // Apply f to the Expressions directly contained in this statement.
  def mapExpr(f: Expression => Expression): Statement = this match {
    case s: DefRegister      => s.copy(clock = f(s.clock))
    case s: DefRegisterReset => s.copy(clock = f(s.clock), reset = f(s.reset), init = f(s.init))
    case s: DefNode          => s.copy(value = f(s.value))
    case s: Connect          => s.copy(loc = f(s.loc), expr = f(s.expr))
    case s: Conditionally    => s.copy(pred = f(s.pred))
    case other               => other
  }
  // Apply f to the child Statements of this statement.
  def mapStmt(f: Statement => Statement): Statement = this match {
    case s: Conditionally => s.copy(conseq = s.conseq.map(f), alt = s.alt.map(f))
    case s: OtherStmt     => s.copy(children = s.children.map(f))
    case other            => other
  }
}
sealed trait IsRegister extends Statement { def name: String; def tpe: String }
case class DefRegister(info: String, name: String, tpe: String, clock: Expression) extends IsRegister
case class DefRegisterReset(info: String, name: String, tpe: String, clock: Expression,
                            reset: Expression, init: Expression) extends IsRegister
case class DefWire(info: String, name: String, tpe: String) extends Statement
case class DefNode(info: String, name: String, value: Expression) extends Statement
case class DefInstance(info: String, name: String, module: String) extends Statement
case class Connect(info: String, loc: Expression, expr: Expression) extends Statement
case class Conditionally(info: String, pred: Expression, conseq: Seq[Statement], alt: Seq[Statement]) extends Statement
// Everything we do not model in detail (printf, assert, mem, ...). Keeps the raw text.
case class OtherStmt(info: String, text: String, children: Seq[Statement]) extends Statement

// ---- Modules and circuits ----
case class Port(info: String, direction: String, name: String, tpe: String)
case class DefModule(info: String, name: String, isPublic: Boolean, ports: Seq[Port], body: Seq[Statement]) {
  def mapStmt(f: Statement => Statement): DefModule = copy(body = body.map(f))
}
case class Circuit(main: String, modules: Seq[DefModule], otherDecls: Seq[String] = Nil) {
  def mapModule(f: DefModule => DefModule): Circuit = copy(modules = modules.map(f))
}

// ---- Step 1: split into lines and recover the nesting from the indentation ----
case class Line(indent: Int, text: String, info: String)
case class Node(line: Line, kids: Seq[Node])

val infoRegex = """^(.*?)\s*@\[(.*)\]\s*$""".r

def toLines(text: String): List[Line] =
  text.linesIterator.filter(_.trim.nonEmpty).map { raw =>
    val indent = raw.length - raw.dropWhile(_ == ' ').length
    raw.trim match {
      case infoRegex(code, info) => Line(indent, code, info)
      case code                  => Line(indent, code, "")
    }
  }.toList

// Parses the sibling lines at `indent`; a more indented line starts the children of the line before it.
def toTree(lines: List[Line], indent: Int): (Vector[Node], List[Line]) = {
  var rest = lines
  val siblings = Vector.newBuilder[Node]
  while (rest.nonEmpty && rest.head.indent == indent) {
    val line = rest.head
    val (kids, remaining) =
      if (rest.tail.nonEmpty && rest.tail.head.indent > indent) toTree(rest.tail, rest.tail.head.indent)
      else (Vector.empty[Node], rest.tail)
    siblings += Node(line, kids)
    rest = remaining
  }
  (siblings.result(), rest)
}

// ---- Step 2: expressions ----
// Split on commas that are not nested inside (), [], {} or <>.
def splitTop(s: String): List[String] = {
  var depth = 0
  val parts = scala.collection.mutable.ListBuffer[String]()
  val cur = new StringBuilder
  for (c <- s) c match {
    case '(' | '[' | '{' | '<' => depth += 1; cur += c
    case ')' | ']' | '}' | '>' => depth -= 1; cur += c
    case ',' if depth == 0     => parts += cur.toString.trim; cur.clear()
    case _                     => cur += c
  }
  if (cur.nonEmpty) parts += cur.toString.trim
  parts.toList
}

val literalRegex = """^((?:UInt|SInt)(?:<\d+>)?)\((.*)\)$""".r
val primRegex    = """^([A-Za-z0-9_]+)\((.*)\)$""".r
val intRegex     = """^-?\d+$""".r

def parseExpr(text: String): Expression = text.trim match {
  case literalRegex(tpe, value) => Literal(tpe, value)
  case primRegex(op, args) =>
    val (consts, exprs) = splitTop(args).partition(a => intRegex.matches(a))
    DoPrim(op, exprs.map(parseExpr), consts.map(_.toInt))
  case ref => Reference(ref)
}

// ---- Step 3: statements, ports, modules ----
val portRegex    = """^(input|output) (\S+) : (.*)$""".r
val regRegex     = """^reg (\S+) : (.*)$""".r
val regResetRegex = """^regreset (\S+) : (.*)$""".r
val wireRegex    = """^wire (\S+) : (.*)$""".r
val nodeRegex    = """^node (\S+) = (.*)$""".r
val instRegex    = """^inst (\S+) of (\S+)$""".r
val connectRegex = """^connect (.*)$""".r
val whenRegex    = """^when (.*) :$""".r
val moduleRegex  = """^(public )?module (\S+) :$""".r

// An `else :` node belongs to the `when` right before it, so we convert siblings together.
def toStatements(nodes: Seq[Node]): Seq[Statement] =
  nodes.foldLeft(Vector.empty[Statement]) {
    case (done, Node(Line(_, "else :", _), kids)) =>
      done.last match {
        case w: Conditionally => done.init :+ w.copy(alt = toStatements(kids))
        case _                => sys.error("`else` without `when`")
      }
    case (done, node) => done :+ toStatement(node)
  }

def toStatement(node: Node): Statement = {
  val Node(Line(_, text, info), kids) = node
  text match {
    case regResetRegex(name, rest) =>
      val Seq(tpe, clock, reset, init) = splitTop(rest)
      DefRegisterReset(info, name, tpe, parseExpr(clock), parseExpr(reset), parseExpr(init))
    case regRegex(name, rest) =>
      val Seq(tpe, clock) = splitTop(rest)
      DefRegister(info, name, tpe, parseExpr(clock))
    case wireRegex(name, tpe)       => DefWire(info, name, tpe)
    case nodeRegex(name, value)     => DefNode(info, name, parseExpr(value))
    case instRegex(name, module)    => DefInstance(info, name, module)
    case connectRegex(args) =>
      val Seq(loc, expr) = splitTop(args)
      Connect(info, parseExpr(loc), parseExpr(expr))
    case whenRegex(pred)            => Conditionally(info, parseExpr(pred), toStatements(kids), Nil)
    case other                      => OtherStmt(info, other, toStatements(kids))
  }
}

def toModule(node: Node): DefModule = {
  val Node(Line(_, text, info), kids) = node
  val moduleRegex(pub, name) = text
  val (portNodes, bodyNodes) = kids.partition(k => portRegex.matches(k.line.text))
  val ports = portNodes.map { k =>
    val portRegex(dir, pname, tpe) = k.line.text
    Port(k.line.info, dir, pname, tpe)
  }
  DefModule(info, name, pub != null, ports, toStatements(bodyNodes))
}

// ---- Step 4: the whole file ----
def parseCircuit(firrtl: String): Circuit = {
  // Skip the "FIRRTL version" line and the multi-line annotation block `circuit Top :%[[ ... ]]`.
  val lines = firrtl.linesIterator.dropWhile(!_.startsWith("circuit ")).toList
  val header   = lines.head
  val afterAnnos = if (header.endsWith("%[[")) lines.tail.dropWhile(_.trim != "]]").tail else lines.tail
  val main = header.stripPrefix("circuit ").takeWhile(c => c != ' ' && c != ':')
  val (children, _) = toTree(toLines(afterAnnos.mkString("\n")), 2)
  // `layer` and `extmodule` declarations are not analyzed: we keep their text to serialize it back.
  def rawLines(n: Node, depth: Int): Seq[String] =
    (("  " * depth) + n.line.text + (if (n.line.info.isEmpty) "" else s" @[${n.line.info}]")) +: n.kids.flatMap(rawLines(_, depth + 1))
  val (modules, others) = children.partition(n => moduleRegex.matches(n.line.text))
  Circuit(main, modules.map(toModule), others.flatMap(rawLines(_, 1)))
}

def serialize(e: Expression): String = e match {
  case Reference(name)         => name
  case Literal(tpe, value)     => s"$tpe($value)"
  case DoPrim(op, args, consts) => (args.map(serialize) ++ consts).mkString(s"$op(", ", ", ")")
}

def serialize(s: Statement, indent: Int): Seq[String] = {
  val pad = "  " * indent
  val loc = (i: String) => if (i.isEmpty) "" else s" @[$i]"
  s match {
    case DefRegister(i, n, t, c)          => Seq(s"${pad}reg $n : $t, ${serialize(c)}${loc(i)}")
    case DefRegisterReset(i, n, t, c, r, v) => Seq(s"${pad}regreset $n : $t, ${serialize(c)}, ${serialize(r)}, ${serialize(v)}${loc(i)}")
    case DefWire(i, n, t)                 => Seq(s"${pad}wire $n : $t${loc(i)}")
    case DefNode(i, n, v)                 => Seq(s"${pad}node $n = ${serialize(v)}${loc(i)}")
    case DefInstance(i, n, m)             => Seq(s"${pad}inst $n of $m${loc(i)}")
    case Connect(i, l, r)                 => Seq(s"${pad}connect ${serialize(l)}, ${serialize(r)}${loc(i)}")
    case Conditionally(i, p, c, a)        =>
      Seq(s"${pad}when ${serialize(p)} :${loc(i)}") ++ c.flatMap(serialize(_, indent + 1)) ++
        (if (a.isEmpty) Nil else Seq(s"${pad}else :") ++ a.flatMap(serialize(_, indent + 1)))
    case OtherStmt(i, t, kids)            => Seq(s"$pad$t${loc(i)}") ++ kids.flatMap(serialize(_, indent + 1))
  }
}

def serialize(m: DefModule): Seq[String] = {
  val loc = (i: String) => if (i.isEmpty) "" else s" @[$i]"
  Seq(s"  ${if (m.isPublic) "public " else ""}module ${m.name} :${loc(m.info)}") ++
    m.ports.map(p => s"    ${p.direction} ${p.name} : ${p.tpe}${loc(p.info)}") ++ Seq("") ++
    m.body.flatMap(serialize(_, 2)) ++ Seq("")
}

def serialize(c: Circuit): String =
  (Seq(s"circuit ${c.main} :") ++ c.otherDecls ++ c.modules.flatMap(serialize)).mkString("\n")

// Pretty-prints any of our case classes as an indented tree: one node per line.
def showTree(x: Any, depth: Int = 0, label: String = ""): String = {
  val pad    = "  " * depth + (if (label.isEmpty) "" else label + ": ")
  def isNode(v: Any): Boolean = v match {
    case _: Expression | _: Statement | _: Port | _: DefModule | _: Circuit => true
    case _ => false
  }
  x match {
    case p: Product if isNode(p) =>
      val fields = p.productElementNames.toList.zip(p.productIterator.toList)
      val (kids, atoms) = fields.partition {
        case (_, v: Seq[_]) => v.nonEmpty && v.forall(isNode)
        case (_, v)         => isNode(v)
      }
      val head = pad + p.productPrefix + atoms.filter(_._2 != "").map { case (n, v) => s" $n=$v" }.mkString
      (head +: kids.flatMap {
        case (n, v: Seq[_]) => v.map(showTree(_, depth + 1, n))
        case (n, v)         => Seq(showTree(v, depth + 1, n))
      }).mkString("\n")
    case other => pad + other.toString
  }
}

// All statements of a body, including the ones nested inside `when`s (pre-order).
def allStatements(body: Seq[Statement]): Seq[Statement] = body.flatMap {
  case w: Conditionally => w +: (allStatements(w.conseq) ++ allStatements(w.alt))
  case o: OtherStmt     => o +: allStatements(o.children)
  case s                => Seq(s)
}

// The Expressions that appear directly in a statement (not the sub-expressions).
def directExprs(s: Statement): Seq[Expression] = s match {
  case r: DefRegister      => Seq(r.clock)
  case r: DefRegisterReset => Seq(r.clock, r.reset, r.init)
  case n: DefNode          => Seq(n.value)
  case c: Connect          => Seq(c.loc, c.expr)
  case w: Conditionally    => Seq(w.pred)
  case _                   => Nil
}

// An Expression and all its sub-expressions (pre-order).
def allExprs(e: Expression): Seq[Expression] = e match {
  case DoPrim(_, args, _) => e +: args.flatMap(allExprs)
  case leaf               => Seq(e)
}

We will also use a small design, with two modules, where `Top` instantiates `Counter`:

In [ ]:
class Counter(width: Int) extends Module {
  val io = IO(new Bundle {
    val en    = Input(Bool())
    val count = Output(UInt(width.W))
  })
  val c = RegInit(0.U(width.W))
  when (io.en) { c := c + 1.U }
  io.count := c
}

class Top extends Module {
  val io = IO(new Bundle {
    val en  = Input(Bool())
    val out = Output(UInt(8.W))
  })
  val fast = Module(new Counter(8))
  val slow = Module(new Counter(4))
  val last = Reg(Bool())
  fast.io.en := io.en
  last := fast.io.count(7)
  slow.io.en := last
  io.out := fast.io.count + slow.io.count
}

lazy val topAST = parseCircuit(getFirrtl(new Top))
println(topAST.main + ": " + topAST.modules.map(_.name))

### Collecting information
Most analyses are *read-only*: they walk the tree and fill some data structure. We have seen in Module 4.2 how to collect the registers of a module. Two more examples follow.

#### Find all connections to a signal
Which statements drive `io.out` in `Top`? In FIRRTL, signals are referenced by *text*, for example `io.out`, `fast.io.count` or `v[1]`; it is up to us to decide what we consider "the same signal". Here, we compare the whole text of the reference. Note how we have to look inside the `when` bodies as well, which `allStatements` does for us:

In [ ]:
def connectsTo(m: DefModule, signal: String): Seq[Connect] =
  allStatements(m.body).collect { case c @ Connect(_, Reference(`signal`), _) => c }

lazy val topModule = topAST.modules.find(_.name == "Top").get
connectsTo(topModule, "io.out").foreach(c => println(serialize(c, 0).head))

The back-quotes in `Reference(`signal`)` make Scala compare with the *value* of `signal`, instead of introducing a new variable called `signal` in the pattern.

Sometimes we want the opposite: a signal's **fan-out**, i.e., all the places where it is *read*. A read is a `Reference` somewhere in an expression, except the left-hand side of a connect:

In [ ]:
def readsOf(m: DefModule, signal: String): Seq[String] =
  allStatements(m.body).flatMap {
    case Connect(_, _, rhs) => allExprs(rhs).collect { case Reference(`signal`) => serialize(rhs) }
    case DefNode(_, name, value) => allExprs(value).collect { case Reference(`signal`) => s"node $name = ${serialize(value)}" }
    // clocks, resets, `when` predicates...
    case s => directExprs(s).flatMap(e => allExprs(e).collect { case Reference(`signal`) => serialize(e) })
  }

println(readsOf(topModule, "fast.io.count"))

Be careful: the Chisel expression `fast.io.count(7)` appears in the FIRRTL as `bits(fast.io.count, 7, 7)` inside of a node, which is why it is found among the `DefNode`s. The Chisel source and the FIRRTL differ, and it is the FIRRTL that you analyze here.

#### The module-instance graph
Which module instantiates which? Every `DefInstance` in a module body is an edge of the hierarchy. Because FIRRTL modules are never nested, this is a graph over module names (and, since a module cannot instantiate itself, directly or not, a DAG):

In [ ]:
// module name -> names of the modules it instantiates (with repetition: an instance for each element)
def instanceGraph(c: Circuit): Map[String, Seq[String]] =
  c.modules.map { m =>
    m.name -> allStatements(m.body).collect { case DefInstance(_, _, module) => module }
  }.toMap

println(instanceGraph(topAST))

A tree view makes the hierarchy easier to see. Recursion on the graph visits `Top`, then each of the modules that it instantiates:

In [ ]:
def printHierarchy(c: Circuit): Unit = {
  val instNames = c.modules.map(m => m.name -> allStatements(m.body).collect { case d: DefInstance => d }).toMap
  def go(module: String, instName: String, depth: Int): Unit = {
    println("  " * depth + s"$instName : $module")
    // An `extmodule` (BlackBox) is not parsed: it is a leaf
    instNames.getOrElse(module, Nil).foreach(i => go(i.module, i.name, depth + 1))
  }
  go(c.main, c.main, 0)
}
printHierarchy(topAST)

And since the graph tells who uses whom, modules that nobody instantiates (other than the top) are dead code:

In [ ]:
val used = instanceGraph(topAST).values.flatten.toSet + topAST.main
println("unused modules: " + topAST.modules.map(_.name).filterNot(used))

### Looking at Chisel instead of FIRRTL
For *generator-time* introspection, there is `DataMirror`, which we met in Module 4.1. For example, inside of `Top`, we could have checked the port list of a child before wiring it up. The FIRRTL view is better when you need the whole design after elaboration; the Chisel view is better when you need to *act* during elaboration. Use whichever fits your problem.

In [ ]:
class CheckedTop extends Module {
  val io = IO(new Bundle { val out = Output(UInt(8.W)) })
  val child = Module(new Counter(8))
  child.io.en := true.B
  io.out := child.io.count
  // Which ports of `child` have a width of exactly 8 bits?
  val wide = DataMirror.fullModulePorts(child).filter { case (_, d) => d.widthOption.contains(8) }
  println("8-bit ports of child: " + wide.map(_._1))
}
getFirrtl(new CheckedTop)
()

### Adding statements
Suppose we want to write a pass that splits nested `DoPrim` expressions, thus transforming this:
```
circuit Top :
  module Top :
    input x : UInt<3>
    input y : UInt<3>
    input z : UInt<3>
    output o : UInt<5>
    connect o, add(x, add(y, z))
```
into this:
```
circuit Top :
  module Top :
    input x : UInt<3>
    input y : UInt<3>
    input z : UInt<3>
    output o : UInt<5>
    node _GEN_0 = add(y, z)
    connect o, add(x, _GEN_0)
```
Our parser reads FIRRTL from any source, also from a string that we write by hand, so we can test on exactly this:

In [ ]:
val nested = """circuit Top :
  module Top :
    input x : UInt<3>
    input y : UInt<3>
    input z : UInt<3>
    output o : UInt<5>
    connect o, add(x, add(y, z))
"""
println(showTree(parseCircuit(nested).modules.head.body.head))

We first need to traverse the AST to every `Statement` and `Expression`. Then, when we see a `DoPrim` *inside of another expression*, we need to add a new `DefNode` before the statement, and insert a reference to that `DefNode` in place of the `DoPrim`. The code below implements this (and preserves the source locator). It uses a *post-order* traversal of expressions: children are split first, so the innermost operation gets the first new name.

Note that a statement is replaced by several statements: for this reason, the function that we map over the statements returns a `Seq[Statement]` and we use `flatMap`.

In [ ]:
object Splitter {
  /** Run splitM on every module */
  def run(c: Circuit): Circuit = c.mapModule(splitM)

  /** Run splitS on the body of the module, using a fresh name generator */
  def splitM(m: DefModule): DefModule = {
    var counter = 0
    // Skip the names that the module already uses
    val used = serialize(m).flatMap("[A-Za-z_][A-Za-z0-9_]*".r.findAllIn(_)).toSet
    def newName(): String = {
      while (used.contains(s"_GEN_$counter")) counter += 1
      val n = s"_GEN_$counter"; counter += 1; n
    }
    m.copy(body = splitBody(m.body, newName))
  }

  def splitBody(body: Seq[Statement], newName: () => String): Seq[Statement] = body.flatMap { s =>
    // Statements nested in a `when` are handled recursively
    val s1 = s match {
      case w: Conditionally => w.copy(conseq = splitBody(w.conseq, newName), alt = splitBody(w.alt, newName))
      case other            => other
    }
    val newNodes = scala.collection.mutable.ArrayBuffer[Statement]()
    // The top expression of a node or a connect may stay a DoPrim: we split only the nested ones.
    val s2 = s1.mapExpr(e => e.mapExpr(splitE(newNodes, newName, s1.info)))
    newNodes.toSeq :+ s2
  }

  /** Post-order: split the children first. If e is a DoPrim, add a new DefNode and return a reference to it. */
  def splitE(nodes: scala.collection.mutable.ArrayBuffer[Statement], newName: () => String, info: String)(e: Expression): Expression =
    e.mapExpr(splitE(nodes, newName, info)) match {
      case p: DoPrim =>
        val name = newName()
        nodes += DefNode(info, name, p)
        Reference(name)
      case other => other
    }
}

println(serialize(Splitter.run(parseCircuit(nested))))

### Deleting statements
Suppose we want to write a pass that inlines all `DefNode`s whose value is a literal, thus transforming this:
```
    node y = UInt<3>(0h1)
    connect o, add(x, y)
```
into this:
```
    connect o, add(x, UInt<3>(0h1))
```
We first need to traverse the AST to every `Statement` and `Expression`. Then, when we see a `DefNode` pointing to a `Literal`, we store it into a hash map and delete the `DefNode` from the module. Then, whenever we see a reference to the deleted node, we insert the corresponding literal.

In [ ]:
object Inliner {
  /** Run inlineM on every module */
  def run(c: Circuit): Circuit = c.mapModule(inlineM)

  def inlineM(m: DefModule): DefModule = {
    val values = scala.collection.mutable.HashMap[String, Expression]()
    // We do not rewrite the text of the statements we did not parse (printf, assert...), so a node
    // that they mention must stay
    val mentioned = allStatements(m.body).collect { case o: OtherStmt => o.text }
      .flatMap("[A-Za-z_][A-Za-z0-9_]*".r.findAllIn(_)).toSet
    m.copy(body = inlineBody(m.body, values, mentioned))
  }

  /** Statements are processed in order, so a node is known before the statements that use it. A DefNode
    * containing a literal is recorded and dropped (returned as an empty Seq), the others are kept. */
  def inlineBody(body: Seq[Statement], values: scala.collection.mutable.HashMap[String, Expression],
                 mentioned: Set[String]): Seq[Statement] =
    body.flatMap { s =>
      s.mapExpr(inlineE(values)) match {
        case DefNode(_, name, l: Literal) if !mentioned(name) =>
          values(name) = l
          Nil // deleted
        case w: Conditionally =>
          Seq(w.copy(conseq = inlineBody(w.conseq, values, mentioned), alt = inlineBody(w.alt, values, mentioned)))
        case other => Seq(other)
      }
    }

  /** If e is a reference whose name is contained in values, return values(name);
    * otherwise run inlineE on all children expressions. */
  def inlineE(values: scala.collection.mutable.HashMap[String, Expression])(e: Expression): Expression = e match {
    case Reference(name) if values.contains(name) => values(name)
    case _                                        => e.mapExpr(inlineE(values))
  }
}

val withLiteral = """circuit Top :
  module Top :
    input x : UInt<3>
    output o : UInt<4>
    node y = UInt<3>(0h1)
    connect o, add(x, y)
"""
println(serialize(Inliner.run(parseCircuit(withLiteral))))

The last two sections are, in the Scala FIRRTL compiler of earlier Chisel versions, the "Add a primop" and the "Swap a statement" idioms. With our small AST they are also simple: adding a primop is a new case in `parseExpr` (it already accepts any lower-case name), and swapping a statement is a `map` that returns a different statement.

<span style="color:red">**Exercise: Find dead nodes**</span><br>
A `node` that is never read is dead code. `firtool` removes it anyway, but in a generator it is often a sign of a bug. Write `unusedNodes`, which returns the names of all the `DefNode`s of a module whose names are never used in any expression of the module (including `when` conditions and the right-hand sides of connects).

In [ ]:
def unusedNodes(m: DefModule): Seq[String] = ???

val deadCode = """circuit Top :
  module Top :
    input x : UInt<3>
    output o : UInt<4>
    node a = add(x, x)
    node b = not(x)
    node c = and(a, a)
    connect o, a
"""
println(unusedNodes(parseCircuit(deadCode).modules.head))
assert(unusedNodes(parseCircuit(deadCode).modules.head) == Seq("b", "c"))
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong> (click to toggle displaying)</label>
<article>
<pre style="background-color:#f7f7f7">
def unusedNodes(m: DefModule): Seq[String] = {
  val stmts = allStatements(m.body)
  val used = stmts.flatMap(directExprs).flatMap(allExprs).collect { case Reference(name) => name }.toSet
  stmts.collect { case DefNode(_, name, _) if !used(name) => name }
}
</pre></article></div></section></div>

Notice that `c` is dead too, but it is `a` that is used by it as well as by the connect. A transitive version (remove `c`, and then see what else became dead) is a nice extension if you want to practice.

That's it for this section! In the next section, we will write a complete analysis of a design.

# You're done!

[Return to the top.](#top)